In [1]:
# ---------- CELL 0: locate the project (same in every notebook) ----------
import os, sys
from pathlib import Path

def find_root(marker="PROJECT_ROOT.marker"):
    here = Path.cwd().resolve()                      # start where this notebook runs
    for folder in [here, *here.parents]:             # go UP one level at a time
        if (folder / marker).exists():
            return folder
    if "google.colab" in sys.modules:                # Colab: look inside Google Drive
        from google.colab import drive
        drive.mount("/content/drive")
        for pattern in (marker, f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}"):
            hits = sorted(Path("/content/drive/MyDrive").glob(pattern))
            if hits:
                return hits[0].parent
    raise FileNotFoundError("PROJECT_ROOT.marker not found - finish Day 1 first.")

ROOT = find_root()
os.chdir(ROOT / "notebooks")
RAW, CLEAN = ROOT/"data"/"raw", ROOT/"data"/"clean"
FEATURES, MODEL_OUT = ROOT/"data"/"features", ROOT/"data"/"model_outputs"
SQL_EXPORTS, BI_READY = ROOT/"data"/"sql_exports", ROOT/"data"/"bi_ready"
MODELS, REPORTS, DOCS = ROOT/"models", ROOT/"reports", ROOT/"docs"
FIGURES = REPORTS/"figures"
sys.path.append(str(ROOT/"src"))
print("Project root:", ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/Internmo/netflix_churn_project


In [2]:
# ---------- CELL 1: write the reusable helper file ----------
helper_code = '''"""Reusable cleaning helpers for the Netflix project."""
import pandas as pd

# canonical name -> every spelling we found in the raw data (all lower case)
COUNTRY_ALIASES = {
    "United States":        ["united states", "usa", "us", "u.s.a", "u.s.", "america"],
    "India":                ["india", "in", "bharat"],
    "United Kingdom":       ["united kingdom", "uk", "u.k.", "britain", "great britain"],
    "Canada":               ["canada", "ca"],
    "Brazil":               ["brazil", "br"],
    "Germany":              ["germany", "de"],
    "France":               ["france", "fr"],
    "Mexico":               ["mexico", "mx"],
    "Japan":                ["japan", "jp"],
    "Philippines":          ["philippines", "ph"],
    "Indonesia":            ["indonesia", "id"],
    "Italy":                ["italy", "it"],
    "Nigeria":              ["nigeria", "ng"],
    "Spain":                ["spain", "es"],
    "South Africa":         ["south africa", "za"],
    "Australia":            ["australia", "au"],
    "South Korea":          ["south korea", "kr", "s. korea", "s.korea"],
    "Argentina":            ["argentina", "ar"],
    "Poland":               ["poland", "pl"],
    "United Arab Emirates": ["united arab emirates", "uae", "u.a.e", "ae"],
}

DEVICE_ALIASES = {
    "Smart TV":       ["smart tv", "smarttv", "tv"],
    "Mobile":         ["mobile", "smartphone", "phone"],
    "Laptop":         ["laptop", "desktop", "pc"],
    "Tablet":         ["tablet", "ipad"],
    "Gaming Console": ["gaming console", "console", "playstation/xbox"],
}

def _build_lookup(aliases):
    """Turn {'India': ['india','in']} into {'india': 'India', 'in': 'India'}."""
    return {spelling: canonical for canonical, spellings in aliases.items() for spelling in spellings}

COUNTRY_LOOKUP = _build_lookup(COUNTRY_ALIASES)
DEVICE_LOOKUP = _build_lookup(DEVICE_ALIASES)

def standardize(series, lookup):
    """Lower-case + strip each value, then translate it. Blanks stay blank. Unknown spellings stay as they were."""
    cleaned = series.astype("string").str.strip().str.lower()
    translated = cleaned.map(lookup)
    return translated.fillna(series.astype("string").str.strip())

def standardize_country(series):
    return standardize(series, COUNTRY_LOOKUP)

def standardize_device(series):
    return standardize(series, DEVICE_LOOKUP)

def save_clean(df, name, clean_folder):
    """Save a cleaned table. Dates are written as YYYY-MM-DD, blanks stay blank (MySQL turns them into NULL)."""
    path = clean_folder / f"{name}.csv"
    df.to_csv(path, index=False, date_format="%Y-%m-%d")
    print(f"saved {path.name:28s} rows={len(df):>7,}  columns={df.shape[1]}")
    return path
'''
(ROOT / "src").mkdir(exist_ok=True)
(ROOT / "src" / "cleaning_helpers.py").write_text(helper_code)

import importlib, cleaning_helpers
importlib.reload(cleaning_helpers)
from cleaning_helpers import standardize_country, standardize_device, save_clean
print(standardize_country(__import__("pandas").Series(["USA", "u.s.a", "Bharat", "india", None, "U.K."])).tolist())

['United States', 'United States', 'India', 'India', <NA>, 'United Kingdom']


In [3]:
# ---------- CELL 2: load raw tables, create the cleaning log ----------
import pandas as pd
import numpy as np

customers     = pd.read_csv(RAW / "customers.csv")
plans         = pd.read_csv(RAW / "subscription_plans.csv")
subscriptions = pd.read_csv(RAW / "subscriptions.csv")
content       = pd.read_csv(RAW / "content.csv")
churn         = pd.read_csv(RAW / "churn_labels.csv")

cleaning_log = []
def log(table, issue, rows_affected, action):
    """Remember what we changed. This becomes docs/cleaning_log.csv."""
    cleaning_log.append({"table": table, "issue": issue, "rows_affected": int(rows_affected), "action": action})
    print(f"[{table}] {issue}: {int(rows_affected)} rows -> {action}")

print("customers rows before cleaning:", len(customers))

customers rows before cleaning: 8048


In [4]:
# ---------- CELL 3: customers - duplicates and whitespace ----------
# 3a) exact duplicate rows
dupes = customers.duplicated().sum()
customers = customers.drop_duplicates(subset="customer_id", keep="first").reset_index(drop=True)
log("customers", "duplicate customer_id rows", dupes, "dropped (kept first copy)")

# 3b) remove stray spaces in every text column
for col in customers.columns:
    if pd.api.types.is_string_dtype(customers[col]):     # only text columns
        customers[col] = customers[col].str.strip()

# 3c) dates
customers["registration_date"] = pd.to_datetime(customers["registration_date"])
print("customers rows now:", len(customers), "| unique ids:", customers["customer_id"].nunique())

[customers] duplicate customer_id rows: 48 rows -> dropped (kept first copy)
customers rows now: 8000 | unique ids: 8000


In [5]:
# ---------- CELL 4: customers - country, city, language ----------
# 4a) country: many spellings -> one name
before = customers["country"].nunique()
customers["country"] = standardize_country(customers["country"])
after = customers["country"].nunique()
log("customers", f"country spellings ({before} variants)", customers["country"].notna().sum(), f"standardised to {after} countries")

# 4b) missing country -> "Unknown" for now. Day 3 will try to recover it from viewing logins.
missing_country = customers["country"].isna().sum()
customers["country"] = customers["country"].fillna("Unknown")
log("customers", "missing country", missing_country, "set to 'Unknown' (Day 3 recovers many from login_location)")

# 4c) missing city and language
for col in ["city", "preferred_language"]:
    n = customers[col].isna().sum()
    customers[col] = customers[col].fillna("Unknown")
    log("customers", f"missing {col}", n, "set to 'Unknown' (cannot be guessed safely)")

# 4d) state_or_region is blank for every row - keep the column (the schema needs it) but remember it is empty
log("customers", "state_or_region blank for all rows", customers["state_or_region"].isna().sum(), "left empty on purpose")

customers["country"].value_counts().head(10)

[customers] country spellings (83 variants): 7591 rows -> standardised to 20 countries
[customers] missing country: 409 rows -> set to 'Unknown' (Day 3 recovers many from login_location)
[customers] missing city: 483 rows -> set to 'Unknown' (cannot be guessed safely)
[customers] missing preferred_language: 251 rows -> set to 'Unknown' (cannot be guessed safely)
[customers] state_or_region blank for all rows: 8000 rows -> left empty on purpose


,count
country,
United States,1449
India,1251
United Kingdom,613
Canada,419
Unknown,409
Brazil,380
Germany,378
France,314
Mexico,299


In [6]:
# ---------- CELL 5: customers - age ----------
customers["age"] = pd.to_numeric(customers["age"], errors="coerce")

impossible = (customers["age"] < 10) | (customers["age"] > 100)
log("customers", "impossible age (<10 or >100)", impossible.sum(), "set to blank, then imputed")
customers.loc[impossible, "age"] = np.nan

missing_age = customers["age"].isna().sum()
segment_median = customers.groupby("customer_segment")["age"].transform("median")
customers["age"] = customers["age"].fillna(segment_median).round().astype("Int64")
log("customers", "missing or impossible age (total)", missing_age, "filled with the median age of the customer_segment")

print(customers.groupby("customer_segment")["age"].median())
print("age range now:", customers["age"].min(), "to", customers["age"].max())

[customers] impossible age (<10 or >100): 21 rows -> set to blank, then imputed
[customers] missing or impossible age (total): 342 rows -> filled with the median age of the customer_segment
customer_segment
Corporate Bundle    33.0
Family              32.0
Individual          32.0
Student             32.0
Name: age, dtype: Float64
age range now: 13 to 76


In [7]:
# ---------- CELL 6: subscriptions ----------
for col in ["subscription_start_date", "subscription_end_date", "cancellation_date"]:
    subscriptions[col] = pd.to_datetime(subscriptions[col])

# True/False -> 1/0 so MySQL stores it as a number
subscriptions["auto_renew"] = subscriptions["auto_renew"].astype(int)

# check 1: every customer must have exactly ONE open (no end date) subscription row = their current plan
open_rows = subscriptions[subscriptions["subscription_end_date"].isna()].groupby("customer_id").size()
assert (open_rows == 1).all() and len(open_rows) == subscriptions["customer_id"].nunique(), "a customer has 0 or 2 open rows!"
print("OK: every customer has exactly one current subscription row")

# check 2: every subscription belongs to a customer that exists
assert subscriptions["customer_id"].isin(customers["customer_id"]).all()
assert subscriptions["plan_id"].isin(plans["plan_id"]).all()

# check 3: monthly_price in subscriptions is the price AT THAT TIME. It can differ from today's plan price. Not an error.
check = subscriptions.merge(plans[["plan_id", "monthly_price"]], on="plan_id", suffixes=("", "_plan_today"))
different = (check["monthly_price"] != check["monthly_price_plan_today"]).sum()
log("subscriptions", "price differs from today's plan price", different, "kept as is (historical price at the time)")

subscriptions["subscription_status"].value_counts()

OK: every customer has exactly one current subscription row
[subscriptions] price differs from today's plan price: 570 rows -> kept as is (historical price at the time)


,count
subscription_status,
Active,6009
Cancelled,1991
Plan Changed,875


In [8]:
# ---------- CELL 7: churn_labels ----------
churn["churned"] = churn["churned"].astype(int)          # True/False -> 1/0
churn["churn_date"] = pd.to_datetime(churn["churn_date"])

assert churn["customer_id"].is_unique
assert churn["customer_id"].isin(customers["customer_id"]).all()
assert set(customers["customer_id"]) == set(churn["customer_id"]), "customers and churn_labels do not match"

# cross-check against subscriptions: churned = 1 must be the same people as status 'Cancelled' on the current row
current = subscriptions[subscriptions["subscription_end_date"].isna()]
check = current.merge(churn, on="customer_id")
print(pd.crosstab(check["subscription_status"], check["churned"]))

churned                 0     1
subscription_status            
Active               6009     0
Cancelled               0  1991


In [9]:
# ---------- CELL 8: subscription_plans and content ----------
# plans: only 3 rows, nothing to fix. Just confirm.
assert plans["plan_id"].is_unique and len(plans) == 3

# content: check ids and ranges
assert content["content_id"].is_unique
print("content rows:", len(content), "| runtime range:", content["duration_minutes"].min(), "-", content["duration_minutes"].max())
log("content", "no known issues", 0, "kept as is")
log("subscription_plans", "no known issues", 0, "kept as is")
content["content_type"].value_counts()

content rows: 500 | runtime range: 20 - 166
[content] no known issues: 0 rows -> kept as is
[subscription_plans] no known issues: 0 rows -> kept as is


,count
content_type,
Movie,216
TV Show,144
Limited Series,59
Documentary,55
Stand-Up Special,26


In [10]:
# ---------- CELL 9: save clean files + the as-of date ----------
CLEAN.mkdir(parents=True, exist_ok=True)

# the as-of date = the last day of viewing activity in the raw data
last_view = pd.read_csv(RAW / "viewing_activity.csv", usecols=["viewing_date"])["viewing_date"].max()
(CLEAN / "as_of_date.txt").write_text(last_view + "\n")
print("As-of date for the whole project:", last_view)

save_clean(customers,     "customers",          CLEAN)
save_clean(plans,         "subscription_plans", CLEAN)
save_clean(subscriptions, "subscriptions",      CLEAN)
save_clean(content,       "content",            CLEAN)
save_clean(churn,         "churn_labels",       CLEAN)

pd.DataFrame(cleaning_log).to_csv(DOCS / "cleaning_log.csv", index=False)
pd.DataFrame(cleaning_log)

As-of date for the whole project: 2026-06-30
saved customers.csv                rows=  8,000  columns=12
saved subscription_plans.csv       rows=      3  columns=6
saved subscriptions.csv            rows=  8,875  columns=10
saved content.csv                  rows=    500  columns=9
saved churn_labels.csv             rows=  8,000  columns=4


,table,issue,rows_affected,action
0,customers,duplicate customer_id rows,48,dropped (kept first copy)
1,customers,country spellings (83 variants),7591,standardised to 20 countries
2,customers,missing country,409,set to 'Unknown' (Day 3 recovers many from log...
3,customers,missing city,483,set to 'Unknown' (cannot be guessed safely)
4,customers,missing preferred_language,251,set to 'Unknown' (cannot be guessed safely)
5,customers,state_or_region blank for all rows,8000,left empty on purpose
6,customers,impossible age (<10 or >100),21,"set to blank, then imputed"
7,customers,missing or impossible age (total),342,filled with the median age of the customer_seg...
8,subscriptions,price differs from today's plan price,570,kept as is (historical price at the time)
9,content,no known issues,0,kept as is


In [11]:
# ---------- CELL 10: validation (read what we saved, not what is in memory) ----------
check_customers = pd.read_csv(CLEAN / "customers.csv")
check_subs      = pd.read_csv(CLEAN / "subscriptions.csv")
check_churn     = pd.read_csv(CLEAN / "churn_labels.csv")

assert len(check_customers) == 8000 and check_customers["customer_id"].is_unique
assert check_customers["country"].nunique() == 21          # 20 countries + 'Unknown'
assert check_customers["age"].between(10, 100).all()
assert len(check_subs) == 8875 and len(check_churn) == 8000
print("All Day 2 checks passed.")
print("Churn rate:", round(check_churn["churned"].mean() * 100, 1), "%")

All Day 2 checks passed.
Churn rate: 24.9 %


In [12]:
# ---------- CELL 11: log entry ----------
entry = """## Day 2 - Cleaning part 1
- customers: 48 duplicate rows removed (8,048 -> 8,000). Country spellings mapped to 20 countries.
- Missing country/city/language set to 'Unknown'. Impossible/missing ages filled with the median age of the segment.
- subscriptions: dates parsed, auto_renew -> 1/0. Every customer has exactly one open row. Prices left historical.
- churn_labels: churned -> 1/0. Proven consistent with subscription status.
- As-of date saved in data/clean/as_of_date.txt (2026-06-30).
- Helper file: src/cleaning_helpers.py. Cleaning log: docs/cleaning_log.csv.
"""
with open(DOCS / "project_log.md", "a") as f:
    f.write(entry + "\n")
print("log updated")

log updated
